# 64 — Structured Network Theory (NeT)

**Dataset:** Structured Retrospective (task-planning app; N=61 behavioral participants)
**Model family:** Network Theory (NeT)
**Numbering:** Structured models use decade 60-68 (family-code positions mirrored from DS5's 41-48,
shifted +20): 61 StM, 62 DyS, 63 StoP, **64 NeT**, 65 InT, 66 GaT, 67 ABM, 68 SyD
**Blueprint:** `outputs/reports/structured/structured_net_blueprint.md` — read this first; it specifies
every FLAG decision made below and the reasoning behind them
**Output root:** `outputs/model_outputs/structured/network_theory/`

---

## Goal

Build per-participant directed weighted task-state transition networks and estimate network structure
metrics (hub-dependence, self-transition rate, transition entropy, stationary distribution).

## Not the goal (Step 1 scope)

Causal inference about why tasks moved between states. Higher-order Markov models. Community detection.
Cross-dataset entropy comparison against DS5 NeT (different state vocabularies — see FLAG W4).


## Proxy / Limitation Warnings (read before interpreting any result below)

- **FLAG W1 — Left-censoring bias.** 60.7% of tasks have no visible `Task Created` event. Edges from
  their first-observed state onward are real, but entry-state statistics (how tasks are "born") are
  only reliable for the 39.3% non-censored subset — reported separately below.
- **FLAG W2 — Temporal adjacency is not a causal claim.** An edge state_i → state_j means the task was
  observed in state_i, then later observed in state_j. It does not mean state_i caused the transition.
- **FLAG W3 — Markov assumption unverified.** The transition matrix T assumes the next state depends
  only on the current state.
- **FLAG W4 — Small state vocabulary (7 states) bounds all entropy/HHI values into a narrow range**
  relative to DS5's 5-50+ app vocabulary. Do not compare magnitudes across datasets.
- **FLAG W5 — No experimental condition.** `subscription_type_last` (free/pro) is the only available
  grouping variable and is purely observational (selection, not randomization).
- **FLAG W6 — Stationary distribution is a descriptive summary, not a verified long-run equilibrium.**


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import warnings
warnings.filterwarnings("ignore")

FEATURES = PROJECT_ROOT / "outputs" / "features" / "structured"
CLEANED = PROJECT_ROOT / "outputs" / "cleaned"
OUT = PROJECT_ROOT / "outputs" / "model_outputs" / "structured" / "network_theory"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUT)

## 1. Load Data

In [ ]:
beh = pd.read_csv(CLEANED / "structured_behavioral_clean.csv")
beh["event_timestamp_utc"] = pd.to_datetime(beh["event_timestamp_utc"], format="mixed", utc=True)

task_feats = pd.read_csv(FEATURES / "structured_task_lifecycle_features.csv")

print(f"Behavioral events: {len(beh):,}")
print(f"Tasks: {len(task_feats):,}")
print(f"Participants: {beh['participant_id'].nunique()}")

## 2. State-Mapping Function (FLAG NET-S1, NET-S2)

Priority order per event, first match wins:
1. `Task Deleted` -> `deleted`
2. `task_completed` tag -> `completed`
3. `task_uncompleted` tag -> `reopened`
4. `date_changed` or `start_time_changed` tag -> `moved`
5. otherwise -> current `task_type` (`inbox` / `timed` / `all_day`) -- includes Task Created and any
   cosmetic-only update (color/icon/notes/subtasks/duration/title/recurrence/energy_level changed, or
   an empty update_type list). This produces a self-loop when the task's type doesn't change.


In [ ]:
RESCHEDULE_TAGS = {"date_changed", "start_time_changed"}

def event_state(row):
    if row["event_type"] == "Task Deleted":
        return "deleted"
    tags = row["update_type_list_str"].split("|") if pd.notna(row["update_type_list_str"]) else []
    if "task_completed" in tags:
        return "completed"
    if "task_uncompleted" in tags:
        return "reopened"
    if RESCHEDULE_TAGS & set(tags):
        return "moved"
    return row["task_type"]

beh_sorted = beh.sort_values(["task_id", "event_timestamp_utc"]).copy()
beh_sorted["state"] = beh_sorted.apply(event_state, axis=1)

print("State value counts (event-level, before pooling into transitions):")
print(beh_sorted["state"].value_counts())

## 3. Build Per-Task Edge Lists, Pool to Participant Level (FLAG NET-G1, NET-S3)

In [ ]:
edge_records = []
entry_records = []

for task_id, grp in beh_sorted.groupby("task_id", sort=False):
    states = grp["state"].tolist()
    pid = grp["participant_id"].iloc[0]
    has_creation = (grp["event_type"] == "Task Created").any()

    entry_records.append({
        "task_id": task_id,
        "participant_id": pid,
        "entry_state": states[0],
        "has_creation_event": has_creation,
    })

    for s_from, s_to in zip(states[:-1], states[1:]):
        edge_records.append({"participant_id": pid, "task_id": task_id,
                              "state_from": s_from, "state_to": s_to})

edges_df = pd.DataFrame(edge_records)
entry_df = pd.DataFrame(entry_records)

print(f"Total transition observations (all participants pooled): {len(edges_df):,}")
print(f"Tasks contributing >=1 edge: {edges_df['task_id'].nunique():,} / {task_feats.shape[0]:,}")

## 4. Entry-State Distribution (FLAG W1 — censored vs. non-censored reported separately)

In [ ]:
print("Entry state distribution -- NON-CENSORED tasks only (has_creation_event=True, reliable):")
print(entry_df[entry_df["has_creation_event"]]["entry_state"].value_counts(normalize=True).round(4))
print()
print("Entry state distribution -- LEFT-CENSORED tasks (first OBSERVED state, not true origin):")
print(entry_df[~entry_df["has_creation_event"]]["entry_state"].value_counts(normalize=True).round(4))

## 5. Participant-Level Transition Matrices and Eligibility (FLAG NET-T1)

In [ ]:
MIN_TRANSITIONS = 20
MIN_STATIONARY = 50
STATES = ["inbox", "timed", "all_day", "moved", "completed", "reopened", "deleted"]

def build_participant_metrics(pid, grp):
    W = grp.groupby(["state_from", "state_to"]).size().unstack(fill_value=0)
    W = W.reindex(index=STATES, columns=STATES, fill_value=0)
    M = int(W.values.sum())

    n_nodes = int(((W.sum(axis=1) + W.sum(axis=0)) > 0).sum())
    n_edges = int((W.values > 0).sum())

    is_eligible = M >= MIN_TRANSITIONS
    if not is_eligible or M == 0:
        return dict(participant_id=pid, n_nodes=n_nodes, n_edges=n_edges, M_transitions=M,
                    is_net_eligible=False, HHI_net=np.nan, top1_share_net=np.nan,
                    top3_share_net=np.nan, rho_self=np.nan, H_trans=np.nan,
                    is_stationary_eligible=False, H_pi=np.nan,
                    reaches_completed=grp["state_to"].eq("completed").any() or grp["state_from"].eq("completed").any(),
                    reaches_deleted=grp["state_to"].eq("deleted").any() or grp["state_from"].eq("deleted").any())

    s_out = W.sum(axis=1)
    shares = (s_out / M).replace([np.inf, -np.inf], 0).fillna(0)
    HHI = float((shares ** 2).sum())
    sorted_shares = shares.sort_values(ascending=False)
    top1 = float(sorted_shares.iloc[0])
    top3 = float(sorted_shares.iloc[:3].sum())
    rho_self = float(np.trace(W.values) / M)

    T = W.div(s_out.replace(0, np.nan), axis=0).fillna(0)
    H_trans = 0.0
    for s in STATES:
        p_s = shares[s]
        if p_s <= 0:
            continue
        row = T.loc[s]
        row_pos = row[row > 0]
        H_row = float(-(row_pos * np.log2(row_pos)).sum())
        H_trans += p_s * H_row

    is_stat_eligible = False
    H_pi = np.nan
    if M >= MIN_STATIONARY:
        try:
            Tm = T.values
            eigvals, eigvecs = np.linalg.eig(Tm.T)
            idx = np.argmin(np.abs(eigvals - 1.0))
            if abs(eigvals[idx] - 1.0) < 1e-6:
                pi = np.real(eigvecs[:, idx])
                pi = pi / pi.sum()
                if (pi >= -1e-9).all():
                    pi = np.clip(pi, 0, None)
                    pi = pi / pi.sum()
                    pi_pos = pi[pi > 0]
                    H_pi = float(-(pi_pos * np.log2(pi_pos)).sum())
                    is_stat_eligible = True
        except np.linalg.LinAlgError:
            pass

    return dict(
        participant_id=pid, n_nodes=n_nodes, n_edges=n_edges, M_transitions=M,
        is_net_eligible=True, HHI_net=round(HHI, 4), top1_share_net=round(top1, 4),
        top3_share_net=round(top3, 4), rho_self=round(rho_self, 4), H_trans=round(H_trans, 4),
        is_stationary_eligible=is_stat_eligible, H_pi=round(H_pi, 4) if not np.isnan(H_pi) else np.nan,
        reaches_completed=bool((grp["state_to"] == "completed").any() or (grp["state_from"] == "completed").any()),
        reaches_deleted=bool((grp["state_to"] == "deleted").any() or (grp["state_from"] == "deleted").any()),
    )

metrics = [build_participant_metrics(pid, grp) for pid, grp in edges_df.groupby("participant_id")]
net_df = pd.DataFrame(metrics)

# participants with zero transitions (all single-event tasks) get a default row
all_pids = set(beh["participant_id"].unique())
missing_pids = all_pids - set(net_df["participant_id"])
for pid in missing_pids:
    net_df = pd.concat([net_df, pd.DataFrame([dict(
        participant_id=pid, n_nodes=0, n_edges=0, M_transitions=0, is_net_eligible=False,
        HHI_net=np.nan, top1_share_net=np.nan, top3_share_net=np.nan, rho_self=np.nan,
        H_trans=np.nan, is_stationary_eligible=False, H_pi=np.nan,
        reaches_completed=False, reaches_deleted=False)])], ignore_index=True)

print(f"Participants: {len(net_df)}")
print(f"NeT-eligible (M>=20): {net_df['is_net_eligible'].sum()} ({net_df['is_net_eligible'].mean()*100:.1f}%)")
print(f"Stationary-eligible (M>=50, ergodic): {net_df['is_stationary_eligible'].sum()} ({net_df['is_stationary_eligible'].mean()*100:.1f}%)")

## 6. Sanity Checks

In [ ]:
eligible = net_df[net_df["is_net_eligible"]]
print("HHI_net in [0,1]:", eligible["HHI_net"].between(0, 1).all())
print("rho_self in [0,1]:", eligible["rho_self"].between(0, 1).all())
print("H_trans in [0, log2(7)=2.807]:", eligible["H_trans"].between(0, np.log2(7) + 1e-6).all())
print()
print(eligible[["n_nodes", "n_edges", "M_transitions", "HHI_net", "rho_self", "H_trans"]].describe().round(3))

## 7. Join Subscription Group (FLAG W5 — observational, not causal)

In [ ]:
sub_last = beh.sort_values("event_timestamp_utc").groupby("participant_id")["subscription_type_at_event"].last()
net_df["subscription_type_last"] = net_df["participant_id"].map(sub_last)
eligible = net_df[net_df["is_net_eligible"]].copy()  # refresh: now includes subscription_type_last

sub_summary = eligible.groupby("subscription_type_last")[
    ["HHI_net", "rho_self", "H_trans", "top1_share_net"]
].agg(["mean", "median", "count"])
print(sub_summary.round(3))
print()
print("NOTE (FLAG W5): purely observational comparison. Free/pro is not randomized -- any")
print("difference could reflect selection (who upgrades), not a causal effect of subscription tier.")

## 8. Task-Outcome Reachability by Network Metric

In [ ]:
corr_hhi_completed = eligible.groupby("reaches_completed")["HHI_net"].mean()
print("Mean HHI_net by whether participant's network ever reaches 'completed':")
print(corr_hhi_completed.round(3))
print()
corr_rho_deleted = eligible.groupby("reaches_deleted")["rho_self"].mean()
print("Mean rho_self by whether participant's network ever reaches 'deleted':")
print(corr_rho_deleted.round(3))

## 9. Figure 1 — HHI_net and H_trans Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(eligible["HHI_net"].dropna(), bins=15, color="#4C72B0", edgecolor="white")
axes[0].set_title("HHI_net (state-flow concentration)")
axes[0].set_xlabel("HHI_net")
axes[1].hist(eligible["H_trans"].dropna(), bins=15, color="#DD8452", edgecolor="white")
axes[1].set_title("H_trans (transition entropy, bits)")
axes[1].set_xlabel("H_trans")
plt.tight_layout()
plt.savefig(FIG / "fig1_net_hhi_h_trans_distribution.png", dpi=150)
plt.show()
print("Saved fig1_net_hhi_h_trans_distribution.png")

## 10. Figure 2 — Pooled State-Transition Graph (all participants combined)

In [ ]:
pooled_W = edges_df.groupby(["state_from", "state_to"]).size().unstack(fill_value=0).reindex(
    index=STATES, columns=STATES, fill_value=0)

G = nx.DiGraph()
for s in STATES:
    G.add_node(s)
for s_from in STATES:
    for s_to in STATES:
        w = pooled_W.loc[s_from, s_to]
        if w > 0:
            G.add_edge(s_from, s_to, weight=w)

pos = nx.circular_layout(G)
weights = [G[u][v]["weight"] for u, v in G.edges()]
max_w = max(weights)

# node_size/font_size increased (1800->5200, 9->13) so labels stay inside the
# circles at the larger size -- see scripts/regenerate_structured_net_pooled_graph.py,
# which is the actual script that produced the current fig2_net_pooled_state_graph.png
# (re-aggregates the frozen transition_edge_list.csv rather than rebuilding from
# raw events here).
fig, ax = plt.subplots(figsize=(9, 9))
nx.draw_networkx_nodes(G, pos, node_color="#4C72B0", node_size=5200, ax=ax)
nx.draw_networkx_labels(G, pos, font_color="white", font_size=13, font_weight="bold", ax=ax)
for (u, v), w in zip(G.edges(), weights):
    nx.draw_networkx_edges(G, pos, edgelist=[(u, v)], width=1 + 4 * w / max_w,
                            alpha=0.6, connectionstyle="arc3,rad=0.15",
                            arrowsize=18, ax=ax)
ax.set_title("Pooled task-state transition network (all 61 participants, edge width ~ log flow)",
              fontsize=17, pad=16)
ax.axis("off")
plt.tight_layout()
plt.savefig(FIG / "fig2_net_pooled_state_graph.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved fig2_net_pooled_state_graph.png")
print()
print("Pooled transition matrix (row=from, col=to):")
print(pooled_W)

## 11. Figure 3 — Subscription Group Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, ["HHI_net", "rho_self", "H_trans"]):
    data = [eligible[eligible["subscription_type_last"] == g][metric].dropna() for g in ["free", "pro"]]
    ax.boxplot(data, labels=["free", "pro"])
    ax.set_title(metric)
plt.suptitle("Subscription group comparison (observational -- FLAG W5)")
plt.tight_layout()
plt.savefig(FIG / "fig3_net_subscription_group_comparison.png", dpi=150)
plt.show()
print("Saved fig3_net_subscription_group_comparison.png")

## 12. Save Outputs

In [ ]:
net_df.to_csv(OUT / "participant_network_metrics.csv", index=False)
edges_df.groupby(["participant_id", "state_from", "state_to"]).size().reset_index(name="count").to_csv(
    OUT / "transition_edge_list.csv", index=False)
entry_df.to_csv(OUT / "entry_state_distribution.csv", index=False)
sub_summary.to_csv(OUT / "subscription_group_summary.csv")

print("Saved:")
for f in ["participant_network_metrics.csv", "transition_edge_list.csv",
          "entry_state_distribution.csv", "subscription_group_summary.csv"]:
    print(" ", f)

## 13. model_run_summary.md

In [ ]:
summary = f"""# Structured NeT — Model Run Summary

**Run date:** 2026-08-03
**Notebook:** notebooks/64_structured_network_theory.ipynb
**Blueprint:** outputs/reports/structured/structured_net_blueprint.md

## Eligibility
- Participants: {len(net_df)}
- NeT-eligible (M>=20 transitions): {int(net_df['is_net_eligible'].sum())} ({net_df['is_net_eligible'].mean()*100:.1f}%)
- Stationary-eligible (M>=50, ergodic): {int(net_df['is_stationary_eligible'].sum())} ({net_df['is_stationary_eligible'].mean()*100:.1f}%)

## Key results (eligible participants, N={int(eligible.shape[0])})
- HHI_net: mean={eligible['HHI_net'].mean():.3f}, median={eligible['HHI_net'].median():.3f}
- rho_self: mean={eligible['rho_self'].mean():.3f}, median={eligible['rho_self'].median():.3f}
- H_trans: mean={eligible['H_trans'].mean():.3f} bits (max possible log2(7)=2.807)
- top1_share_net: mean={eligible['top1_share_net'].mean():.3f}

## Entry-state distribution (non-censored tasks only, FLAG W1)
{entry_df[entry_df['has_creation_event']]['entry_state'].value_counts(normalize=True).round(4).to_string()}

## Subscription group comparison (FLAG W5 -- observational only)
{sub_summary.round(3).to_string()}

## Flags applied
See outputs/reports/structured/structured_net_blueprint.md Part 2-4 for full FLAG rationale:
NET-S1 (state priority order), NET-S2 (cosmetic edits = self-loop), NET-S3 (self-loops included),
NET-G1 (task-then-participant pooling), NET-T1 (MIN_TRANSITIONS=20/50), W1-W6 (weakness flags).
"""

with open(OUT / "model_run_summary.md", "w") as f:
    f.write(summary)

print(summary)